# Sequential MNIST Tutorial

Training a GLIF RSNN on Sequential MNIST with threshold-based encoding.

**Task:** Classify MNIST digits presented pixel-by-pixel in sequence (784 timesteps).

**Key features:**
- 80 input neurons with threshold encoding
- 220 recurrent neurons (paper configuration)
- 100 neurons with spike frequency adaptation (SFA)
- Low-pass filter readout

In [ ]:
import torch
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from btorch.models import environ, functional
from btorch.models.init import uniform_v_

from src.model import SingleLayerGLIFRSNN
from src.loss import CombinedLoss, LossConfig
from src.checkpoint import save_checkpoint, load_checkpoint
from src.calibration import calibrate_io_scales
from tutorials.seqmnist import get_dataloaders, get_task_defaults

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Load Data

In [ ]:
# Get task defaults (paper configuration)
defaults = get_task_defaults()
print("Task defaults:")
for k, v in defaults.items():
    print(f"  {k}: {v}")

# Simple config object
class Config:
    pass

config = Config()
for k, v in defaults.items():
    setattr(config, k, v)
config.data_dir = '../data'

# Load dataloaders
train_loader, test_loader, input_dim, output_dim, T = get_dataloaders(config)

print(f"\nData loaded:")
print(f"  Input dim: {input_dim}")
print(f"  Output dim: {output_dim}")
print(f"  Timesteps: {T}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 2. Visualize Threshold Encoding

In [ ]:
# Get a sample
x_batch, y_batch = next(iter(train_loader))
print(f"Batch shape: {x_batch.shape}")  # (T, batch, 80)

# Plot first sample
x_sample = x_batch[:, 0, :].cpu()  # (T, 80)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot spike raster
ax = axes[0]
for neuron_idx in range(80):
    spike_times = torch.where(x_sample[:, neuron_idx] > 0)[0]
    ax.scatter(spike_times.numpy(), [neuron_idx] * len(spike_times), 
               c='blue', s=2, alpha=0.6)
ax.set_xlabel('Time (pixel index)')
ax.set_ylabel('Input Neuron')
ax.set_title(f'Input Spike Raster (Label: {y_batch[0].item()})')
ax.set_xlim(0, T)

# Plot pixel values
ax = axes[1]
# Reconstruct approximate pixel values from spike pattern
pixel_vals = torch.zeros(T)
thresholds = torch.linspace(0, 1, 80)
for t in range(T):
    active = x_sample[t] > 0
    if active.any():
        pixel_vals[t] = thresholds[active].mean()
ax.plot(pixel_vals.numpy(), 'k-', linewidth=0.5)
ax.set_xlabel('Time (pixel index)')
ax.set_ylabel('Approximate Pixel Value')
ax.set_title('Pixel Value Trajectory')
ax.set_xlim(0, T)

plt.tight_layout()
plt.show()

## 3. Create Model

In [ ]:
model = SingleLayerGLIFRSNN(
    input_dim=input_dim,
    output_dim=output_dim,
    n_neuron=defaults['n_neuron'],
    n_adapt=defaults['n_adapt'],
    asc_amp=defaults['asc_amp'],
    tau_adapt=defaults['tau_adapt'],
    tau_mem=defaults['tau_mem'],
    v_threshold=defaults['v_threshold'],
    dt=defaults['dt'],
).to(device)

print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Neurons: {model.n_neuron}, Adaptive: {model.n_adapt}")

## 4. Training with State Management

In [ ]:
# Initialize state
functional.init_net_state(model.rnn, batch_size=config.batch_size, device=device)
uniform_v_(model.neuron, set_reset_value=True)

# Setup training
optimizer = torch.optim.Adam(model.parameters(), lr=defaults['lr'])
loss_fn = CombinedLoss(LossConfig(rate_weight=defaults['rate_weight']), dt=defaults['dt'])

# Training loop
num_epochs = 5  # Small for demo

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (x, target) in enumerate(train_loader):
        x, target = x.to(device), target.to(device)
        
        # Reset state
        functional.reset_net(model.rnn, batch_size=x.shape[1])
        
        optimizer.zero_grad()
        
        # Forward with dt context
        with environ.context(dt=defaults['dt']):
            output, states = model(x)
        
        # Loss
        loss, _ = loss_fn(output, target, states, T)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        # Stats
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)
        
        if batch_idx >= 10:  # Just a few batches for demo
            break
    
    acc = 100. * correct / total
    print(f"Epoch {epoch+1}: loss={total_loss/11:.4f}, acc={acc:.2f}%")

## 5. Visualization

In [ ]:
model.eval()

with torch.no_grad():
    x, target = next(iter(test_loader))
    x = x.to(device)
    
    functional.reset_net(model.rnn, batch_size=x.shape[1])
    
    with environ.context(dt=defaults['dt']):
        output, states = model(x)
    
    spikes = states['spikes'].cpu()[:, 0, :]  # First sample
    pred = output[0].argmax().item()

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

n_show = 100
for neuron_idx in range(min(n_show, model.n_neuron)):
    spike_times = torch.where(spikes[:, neuron_idx] > 0)[0]
    color = 'red' if neuron_idx < defaults['n_adapt'] else 'black'
    ax.scatter(spike_times.numpy(), [neuron_idx] * len(spike_times), 
               c=color, s=5, alpha=0.6)

ax.axhline(y=defaults['n_adapt'], color='blue', linestyle='--', alpha=0.5, label='Adaptive boundary')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Neuron Index')
ax.set_title(f'Recurrent Layer Spikes (True: {target[0].item()}, Pred: {pred})')
ax.legend()
plt.tight_layout()
plt.show()